# 🚀 ComfyUI Google Colab - Cài Đặt Trực Tiếp Vào Google Drive

Notebook này tự động cài đặt và vận hành **ComfyUI** trực tiếp trên **Google Drive** của bạn:
- 💾 **Lưu Trữ Vĩnh Viễn Trên Google Drive**:
  - **Mã Nguồn & Models**: Lưu tại `/content/drive/MyDrive/ComfyUI` (tất cả Checkpoints, LoRA, ControlNet, Custom Nodes lưu trên Drive, lần sau mở lên dùng ngay không cần tải lại).
  - **Thư Mục Ảnh Riêng Biệt**: Lưu tại `/content/drive/MyDrive/ComfyUI_Outputs` (toàn bộ ảnh tạo ra / ảnh train được tự động lưu vào thư mục riêng này).
- 📦 **Script Tự Động Nâng Cấp**: Sử dụng bộ script `install.sh` tối ưu (tự bổ sung công cụ, cơ chế mirror fallback cho Civitai/HuggingFace).
- 🌐 **Đường Hầm Kết Nối Đa Nguồn (Dual-Tunnel)**: Chạy song song **Cloudflare Tunnel** và **Localtunnel** dự phòng.

👉 **Hướng dẫn sử dụng**: Chọn **Runtime -> Run all** (hoặc bấm tổ hợp phím `Ctrl + F9`) để chạy từ A đến Z!

In [ ]:
# @title 1. Gắn Google Drive & Khởi tạo cấu trúc thư mục lưu trữ vĩnh viễn
from google.colab import drive
import os

print("📁 Đang kết nối với Google Drive...")
drive.mount("/content/drive")

# 1. Thư mục ComfyUI chính trên Google Drive (Lưu Models, Checkpoints, LoRAs, Custom Nodes)
drive_comfy_dir = "/content/drive/MyDrive/ComfyUI"
os.makedirs(drive_comfy_dir, exist_ok=True)

# 2. Thư mục riêng biệt lưu ảnh Output / Ảnh Train trên Google Drive
drive_outputs_dir = "/content/drive/MyDrive/ComfyUI_Outputs"
os.makedirs(drive_outputs_dir, exist_ok=True)

print(f"✅ Thư mục ComfyUI (Models/Nodes) trên Drive: {drive_comfy_dir}")
print(f"✅ Thư mục Lưu Ảnh riêng biệt trên Drive:       {drive_outputs_dir}")


In [ ]:
# @title 2. Cài đặt ComfyUI & Tải toàn bộ Models/Nodes trực tiếp vào Google Drive
import os
import subprocess

%cd /content

# 1. Clone bộ script cài đặt từ GitHub
if not os.path.exists("/content/comfyui-setup1"):
    print("📦 Tải bộ script setup từ GitHub...")
    !git clone https://github.com/hung187/comfyui-setup1.git /content/comfyui-setup1

%cd /content/comfyui-setup1
!git pull

# 2. Chạy install.sh cài đặt ComfyUI + Models + Nodes trực tiếp vào Google Drive
print("🚀 Bắt đầu cài đặt ComfyUI & Tải toàn bộ Models/Nodes vào Google Drive...")
!bash install.sh --comfy-dir "/content/drive/MyDrive/ComfyUI" --civitai-token "63190c338eed6411b6adbcaecef169bc"

# 3. Liên kết (symlink) thư mục output của ComfyUI sang thư mục ComfyUI_Outputs riêng trên Drive
comfy_output_symlink = "/content/drive/MyDrive/ComfyUI/output"
if os.path.exists(comfy_output_symlink) and not os.path.islink(comfy_output_symlink):
    !rm -rf "{comfy_output_symlink}"

if not os.path.exists(comfy_output_symlink):
    !ln -s "/content/drive/MyDrive/ComfyUI_Outputs" "{comfy_output_symlink}"
    print("🔗 Đã liên kết thư mục Output của ComfyUI với thư mục ComfyUI_Outputs trên Drive!")

print("🎉 ĐÃ HOÀN TẤT CÀI ĐẶT TOÀN BỘ DỮ LIỆU VÀO GOOGLE DRIVE!")


In [ ]:
# @title 3. Khởi động ComfyUI từ Drive & Mở Đường Hầm Kết Nối (Cloudflare & Localtunnel)
import os
import time
import subprocess
import re
import urllib.request

# 1. Tải và cài đặt Cloudflared nếu chưa có
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📦 Đang cài đặt Cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    !rm -f cloudflared-linux-amd64.deb
    print("✅ Đã cài đặt Cloudflared!")

# Cài nodejs npm cho localtunnel dự phòng
!apt-get update -qq && apt-get install -y -qq nodejs npm > /dev/null 2>&1

%cd /content/drive/MyDrive/ComfyUI

# 2. Khởi chạy ComfyUI Core trực tiếp từ Google Drive
comfy_cmd = "python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header"
comfy_log = "/content/comfyui.log"

if os.path.exists(comfy_log):
    os.remove(comfy_log)

subprocess.Popen(f"{comfy_cmd} > {comfy_log} 2>&1", shell=True)
print("⏳ Đang khởi động ComfyUI từ Google Drive (vui lòng đợi vài giây cho ComfyUI nạp model).../")

# Polling kiểm tra ComfyUI sẵn sàng tại port 8188
comfy_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/", timeout=2) as response:
            if response.status == 200:
                comfy_ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if comfy_ready:
    print("✅ ComfyUI Server từ Google Drive đã sẵn sàng tại 127.0.0.1:8188!")
else:
    print("⚠️ Khởi động ComfyUI tốn nhiều thời gian hơn dự kiến, tiến hành mở đường hầm...")

# 3. Khởi chạy Cloudflare Tunnel ở background
tunnel_cmd = "cloudflared tunnel --url http://127.0.0.1:8188"
tunnel_log = "/content/cloudflared.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)
subprocess.Popen(f"{tunnel_cmd} > {tunnel_log} 2>&1", shell=True)

# 4. Khởi chạy Localtunnel làm đường hầm dự phòng
lt_log = "/content/localtunnel.log"
if os.path.exists(lt_log):
    os.remove(lt_log)
subprocess.Popen(f"npx localtunnel --port 8188 > {lt_log} 2>&1", shell=True)

# Lấy password giải mã của Localtunnel (IP Public Colab)
try:
    colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf-8').strip()
except Exception:
    colab_ip = "N/A"

# 5. Quét log tìm Public URLs
cf_url = None
lt_url = None

for _ in range(25):
    time.sleep(1)
    if not cf_url and os.path.exists(tunnel_log):
        with open(tunnel_log, "r") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                cf_url = m.group(0)
    if not lt_url and os.path.exists(lt_log):
        with open(lt_log, "r") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", f.read())
            if m:
                lt_url = m.group(0)
    if cf_url and lt_url:
        break

print("\n" + "═"*75)
if cf_url:
    print(f"🌐 LINK CHÍNH (Cloudflare Tunnel): {cf_url}")
else:
    print("🌐 LINK CHÍNH (Cloudflare): Đang kết nối, kiểm tra log dưới.")

if lt_url:
    print(f"🔄 LINK DỰ PHÒNG (Localtunnel):     {lt_url}")
    print(f"🔑 Mật khẩu nhập vào Localtunnel:  {colab_ip}")

print("📁 Mọi ảnh sinh ra / ảnh train được lưu riêng tại: Google Drive -> ComfyUI_Outputs")
print("💡 Nếu mạng Viettel/VNPT/FPT bị chặn trycloudflare.com, hãy dùng Link Dự Phòng!")
print("═"*75 + "\n")

# 6. Live stream log ComfyUI
print("📋 STREAM LOG COMFYUI (Đang hoạt động...):")
try:
    with open(comfy_log, "r") as f:
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng ComfyUI.")
